In [ ]:
import os
import json
from reportlab.lib.pagesizes import letter
from reportlab.lib.styles import getSampleStyleSheet, ParagraphStyle
from reportlab.lib.units import inch
from reportlab.platypus import SimpleDocTemplate, Paragraph, Spacer, PageBreak
from reportlab.lib import colors
from config import *
from utils import *

In [ ]:
def chunk_text(text, chunk_size=CHUNK_SIZE, overlap=CHUNK_OVERLAP):
    """Split text into overlapping chunks."""
    chunks = []
    start = 0
    while start < len(text):
        end = start + chunk_size
        chunks.append(text[start:end])
        start += chunk_size - overlap
    return chunks

In [ ]:
def filter_relevant_chunks(chunks, topic, client, max_tokens_filter=MAX_TOKENS_FILTER):
    """Filter chunks to only those relevant to the given topic."""
    relevant = []
    for i, chunk in enumerate(chunks):
        prompt = f"""Does the following text contain ANY of the following, relevant to: {topic}?

- Federal or state laws or legal statutes
- Procedural rules or requirements officers must follow
- Medical or factual information about crimes or injuries
- Logical dependencies between facts (if X then Y)
- Lists of required documentation elements
- Information about co-occurring crimes
- Conditions under which actions are permitted or prohibited

Answer only YES or NO.

Text:
{chunk}"""

        response = client.chat.completions.create(
            model=MODEL_NAME,
            messages=[{"role": "user", "content": prompt}],
            temperature=TEMP_DETERMINISTIC,
            max_tokens=max_tokens_filter
        )

        answer = response.choices[0].message.content.strip().upper()
        if "YES" in answer:
            relevant.append(chunk)
            print(f"  Chunk {i+1}/{len(chunks)} — relevant")
        else:
            print(f"  Chunk {i+1}/{len(chunks)} — skipped")

    return relevant

In [ ]:
def extract_context_for_topic(topic, source_texts, client):
    """Chunk, filter, and extract legal context rules from source documents."""
    print(f"\nExtracting context for: {topic}")

    all_relevant_chunks = []
    for doc_name, text in source_texts:
        print(f"\n  Scanning {doc_name}...")
        chunks = chunk_text(text)
        relevant = filter_relevant_chunks(chunks, topic, client)
        all_relevant_chunks.extend(relevant)
        print(f"  {len(relevant)}/{len(chunks)} chunks relevant")

    if not all_relevant_chunks:
        print("  Warning: no relevant chunks found")
        return ""

    combined_text = "\n\n---\n\n".join(all_relevant_chunks)

    extraction_prompt = f"""You are extracting legally and procedurally relevant
rules from police training documents for use in a training simulator.
These rules will be used to ensure generated scenarios are legally and
logically consistent.

Topic: {topic}

Pay special attention to extracting:
- Exact legal statutes and what they prohibit or require
- IF/THEN logical rules — if one fact is true, what other facts must
  also be true or false as a result
- Mutually exclusive states — facts that cannot both be true at the same time
- Specific conditions under which actions are permitted or prohibited

Examples of the kind of rules we need:
- "If a qualifying protection order is in place, the suspect is legally
  prohibited from possessing firearms under 18 U.S.C. § 922(g)(8)"
- "Firearms can only be confiscated at the scene if the suspect is present
  and arrested, firearms are in plain view, or a search warrant is obtained"
- "If the suspect was not present when officers arrived, an arrest cannot
  have been made at the scene — a warrant must be sought instead"

From the text below, extract all such rules you can find.
Be specific and concrete. Format as a numbered list.
Do not include general advice or platitudes — only precise actionable rules.
Plain text only, no JSON.

Source text:
{combined_text}"""

    response = client.chat.completions.create(
        model=MODEL_NAME,
        messages=[{"role": "user", "content": extraction_prompt}],
        temperature=TEMP_DETERMINISTIC
    )

    return response.choices[0].message.content.strip()

In [ ]:
def build_legal_context_library(client):
    """Build the legal context library from source manuals. Skips if already exists."""
    os.makedirs(LEGAL_CONTEXT_DIR, exist_ok=True)
    output_path = os.path.join(LEGAL_CONTEXT_DIR, "legal_context.txt")

    if os.path.exists(output_path):
        print("legal_context.txt already exists — skipping")
        return

    print("Loading source documents...")
    source_texts = []
    for filename in os.listdir(MANUALS_DIR):
        if filename.endswith(".pdf"):
            filepath = os.path.join(MANUALS_DIR, filename)
            print(f"  Loading: {filename}")
            text = extract_text_from_pdf(filepath)
            source_texts.append((filename, text))
    print(f"\n{len(source_texts)} documents loaded.")

    topic = (
        "universal legal and procedural rules that apply to all law enforcement "
        "incidents involving intimate partner violence, regardless of specific "
        "crime type. Include: federal firearms prohibition laws, the legal "
        "relationship between protection orders and firearms possession, "
        "firearms confiscation rules based on suspect presence at scene, "
        "arrest vs warrant procedures based on whether suspect is on scene, "
        "and logical dependencies between facts that officers must document consistently."
    )

    context = extract_context_for_topic(topic, source_texts, client)

    with open(output_path, "w") as f:
        f.write(context)

    print(f"\nSaved: {output_path}")

In [ ]:
def build_all_checklists(client):
    """Extract and save all crime type checklists from source PDFs. Skips existing files."""
    os.makedirs(CHECKLIST_JSON_DIR, exist_ok=True)

    for crime_type, filename in CHECKLIST_FILES.items():
        slug = crime_type.lower().replace(" ", "_").replace("/", "_")
        output_path = os.path.join(CHECKLIST_JSON_DIR, f"{slug}.json")

        if os.path.exists(output_path):
            print(f"Skipping {crime_type} — already exists")
            continue

        print(f"Processing {crime_type}...")
        pdf_path = os.path.join(CHECKLIST_DIR, filename)
        raw_text = extract_text_from_pdf(pdf_path)
        checklist = extract_checklist_from_text(raw_text, crime_type, client, MODEL_NAME, TEMP_DETERMINISTIC)

        with open(output_path, "w") as f:
            json.dump(checklist, f, indent=2)

        print(f"  Saved: {output_path}")

    print("\nAll done!")

In [ ]:
def generate_validation_pdf(client, scenarios_per_type=5):
    """Generate a PDF of scenarios per crime type for professor review."""
    from scenario_generator import generate_scenario
    from data_loader import load_checklist

    validation_dir = os.path.join(os.path.dirname(LEGAL_CONTEXT_DIR), "..", "validation")
    os.makedirs(validation_dir, exist_ok=True)
    output_path = os.path.join(validation_dir, "scenarios_for_review.pdf")

    doc = SimpleDocTemplate(
        output_path,
        pagesize=letter,
        leftMargin=0.75*inch,
        rightMargin=0.75*inch,
        topMargin=0.75*inch,
        bottomMargin=0.75*inch
    )

    styles = getSampleStyleSheet()

    title_style = ParagraphStyle(
        'ScenarioTitle',
        parent=styles['Heading1'],
        fontSize=14,
        spaceAfter=6,
        textColor=colors.HexColor('#1a1a2e')
    )
    meta_style = ParagraphStyle(
        'Meta',
        parent=styles['Normal'],
        fontSize=10,
        spaceAfter=4,
        textColor=colors.HexColor('#333333')
    )
    question_style = ParagraphStyle(
        'Question',
        parent=styles['Normal'],
        fontSize=9,
        spaceAfter=1,
        textColor=colors.HexColor('#555555'),
        fontName='Helvetica-Oblique'
    )
    answer_style = ParagraphStyle(
        'Answer',
        parent=styles['Normal'],
        fontSize=9,
        spaceAfter=6,
        leftIndent=12,
        textColor=colors.black
    )
    counter_style = ParagraphStyle(
        'Counter',
        parent=styles['Normal'],
        fontSize=8,
        spaceAfter=12,
        textColor=colors.HexColor('#888888')
    )

    story = []

    for crime_type in CRIME_TYPES:
        print(f"\nGenerating {scenarios_per_type} scenarios for: {crime_type}")
        checklist = load_checklist(crime_type)

        for i in range(scenarios_per_type):
            print(f"  Scenario {i+1}/{scenarios_per_type}...")
            scenario = generate_scenario(checklist)

            story.append(Paragraph(
                f"{crime_type} — Scenario {i+1} of {scenarios_per_type}",
                title_style
            ))
            story.append(Paragraph(
                f"<b>Interviewee:</b> {scenario['interviewee_name']} ({scenario['interviewee_role']})",
                meta_style
            ))
            story.append(Paragraph(
                f"<b>Dispatch:</b> {scenario['dispatch_line']}",
                meta_style
            ))
            story.append(Paragraph(
                f"Scenario {i+1} of {scenarios_per_type} for {crime_type}",
                counter_style
            ))

            for el in checklist["required_elements"]:
                answer = scenario["facts"].get(el["id"], "MISSING")
                question = el["element"].replace("&", "&amp;").replace("<", "&lt;").replace(">", "&gt;")
                answer = str(answer).replace("&", "&amp;").replace("<", "&lt;").replace(">", "&gt;")

                story.append(Paragraph(f"Q: {question}", question_style))
                story.append(Paragraph(f"A: {answer}", answer_style))

            story.append(PageBreak())

    doc.build(story)
    print(f"\nSaved: {output_path}")
    return output_path

In [ ]:
def build_writing_standards_library(client):
    """Build the writing standards library from source manuals. Skips if already exists."""
    os.makedirs(WRITING_STANDARDS_DIR, exist_ok=True)
    output_path = os.path.join(WRITING_STANDARDS_DIR, "writing_standards.txt")

    if os.path.exists(output_path):
        print("writing_standards.txt already exists — skipping")
        return

    print("Loading source documents...")
    source_texts = []
    for filename in os.listdir(MANUALS_DIR):
        if filename.endswith(".pdf"):
            filepath = os.path.join(MANUALS_DIR, filename)
            print(f"  Loading: {filename}")
            text = extract_text_from_pdf(filepath)
            source_texts.append((filename, text))
    print(f"\n{len(source_texts)} documents loaded.")

    topic = (
        "concrete, specific rules for professional police report writing style and standards. "
        "Include: active vs passive voice rules, how to describe observations vs conclusions, "
        "how to attribute statements and hearsay properly, rules about first vs third person, "
        "chronological ordering of events, how to describe suspect behavior using observable facts "
        "rather than conclusions, avoiding vague language, how to document victim and witness "
        "statements, and any other specific writing standards referenced in law enforcement "
        "report writing manuals."
    )

    standards = extract_context_for_topic(topic, source_texts, client)

    with open(output_path, "w") as f:
        f.write(standards)

    print(f"\nSaved: {output_path}")

# Claude sonnet 4.6 rewrote the writing_standards.txt as this function did not
# create one that was detailed enough